In [35]:
import numpy as np

In [36]:
def get_column_space_basis(matrix):
    """
    Find the basis for the column space and left null space of a 3x3 matrix.
    
    Parameters:
        matrix: A 3x3 numpy array (rank 1 or rank 2)
    
    Returns:
        A tuple (col_space_basis, left_null_space_basis) where each is a list of vectors
    """
    A = np.array(matrix, dtype=float)
    
    if A.shape != (3, 3):
        raise ValueError("Matrix must be 3x3")
    
    # Perform row reduction to find pivot columns
    rref, pivot_cols = row_echelon_with_pivots(A.copy())
    
    # Column space basis: original columns at pivot positions
    col_space_basis = [A[:, col] for col in pivot_cols]
    
    # Left null space: null space of A^T
    left_null_basis = get_null_space(A.T)
    
    return col_space_basis, left_null_basis


def row_echelon_with_pivots(A, tol=1e-10):
    """
    Perform Gaussian elimination to get row echelon form.
    Returns the row echelon form and the indices of pivot columns.
    """
    rows, cols = A.shape
    pivot_cols = []
    current_row = 0
    
    for col in range(cols):
        if current_row >= rows:
            break
            
        # Find the pivot (largest absolute value in current column)
        max_row = current_row
        for row in range(current_row + 1, rows):
            if abs(A[row, col]) > abs(A[max_row, col]):
                max_row = row
        
        # Check if pivot is essentially zero
        if abs(A[max_row, col]) < tol:
            continue
            
        # Swap rows
        A[[current_row, max_row]] = A[[max_row, current_row]]
        
        # Record pivot column
        pivot_cols.append(col)
        
        # Eliminate below
        for row in range(current_row + 1, rows):
            if abs(A[row, col]) > tol:
                factor = A[row, col] / A[current_row, col]
                A[row, col:] -= factor * A[current_row, col:]
        
        current_row += 1
    
    return A, pivot_cols


def get_null_space(A, tol=1e-10):
    """
    Find the null space basis of matrix A using RREF.
    For a matrix A (m x n), finds vectors x such that Ax = 0.
    """
    A = A.copy().astype(float)
    rows, cols = A.shape
    
    # Perform RREF
    pivot_cols = []
    current_row = 0
    
    for col in range(cols):
        if current_row >= rows:
            break
            
        # Find pivot
        max_row = current_row
        for row in range(current_row + 1, rows):
            if abs(A[row, col]) > abs(A[max_row, col]):
                max_row = row
        
        if abs(A[max_row, col]) < tol:
            continue
            
        # Swap rows
        A[[current_row, max_row]] = A[[max_row, current_row]]
        pivot_cols.append(col)
        
        # Normalize pivot row
        A[current_row] = A[current_row] / A[current_row, col]
        
        # Eliminate above and below
        for row in range(rows):
            if row != current_row and abs(A[row, col]) > tol:
                A[row] -= A[row, col] * A[current_row]
        
        current_row += 1
    
    # Find free variables (non-pivot columns)
    free_cols = [c for c in range(cols) if c not in pivot_cols]
    
    # Build null space basis vectors
    null_basis = []
    for free_col in free_cols:
        # Create a vector with 1 in the free variable position
        vec = np.zeros(cols)
        vec[free_col] = 1
        
        # Fill in pivot variable values
        for i, pivot_col in enumerate(pivot_cols):
            vec[pivot_col] = -A[i, free_col]
        
        null_basis.append(vec)
    
    return null_basis


def verify_orthogonality(col_basis, left_null_basis, tol=1e-10):
    """
    Verify that column space and left null space are orthogonal.
    
    Parameters:
        col_basis: List of column space basis vectors
        left_null_basis: List of left null space basis vectors
        tol: Tolerance for considering a dot product as zero
    
    Returns:
        True if all pairs are orthogonal, False otherwise
    """
    all_orthogonal = True
    
    print("Checking orthogonality (dot products):")
    for i, col_vec in enumerate(col_basis):
        for j, null_vec in enumerate(left_null_basis):
            dot_product = np.dot(col_vec, null_vec)
            is_orthogonal = abs(dot_product) < tol
            status = "✓" if is_orthogonal else "✗"
            print(f"  col_v{i+1} · null_u{j+1} = {dot_product:.10f} {status}")
            if not is_orthogonal:
                all_orthogonal = False
    
    if all_orthogonal:
        print("\nAll vectors are orthogonal!")
    else:
        print("\nSome vectors are NOT orthogonal!")
    
    return all_orthogonal


def generate_rank_n_matrix(n):
    """
    Generate a random 3x3 integer matrix with rank n.
    
    Parameters:
        n: The desired rank (1 or 2)
    
    Returns:
        A 3x3 numpy array of integers with the specified rank
    """
    if n not in [1, 2]:
        raise ValueError("n must be 1 or 2")
    
    if n == 1:
        # Rank-1: outer product of two vectors (all columns are multiples of one vector)
        col = np.random.randint(-5, 6, size=(3, 1))
        row = np.random.randint(-5, 6, size=(1, 3))
        matrix = col @ row
    else:
        # Rank-2: sum of two rank-1 matrices
        col1 = np.random.randint(-5, 6, size=(3, 1))
        row1 = np.random.randint(-5, 6, size=(1, 3))
        col2 = np.random.randint(-5, 6, size=(3, 1))
        row2 = np.random.randint(-5, 6, size=(1, 3))
        matrix = col1 @ row1 + col2 @ row2
    
    return matrix

In [37]:
# Rank-1 matrix: all rows are multiples of [1, 2, 3]
rank1_matrix = np.array([
    [1, 2, 3],
    [2, 4, 6],
    [3, 6, 9]
])

print("Rank-1 Matrix:")
print(rank1_matrix)
print(f"\nMatrix rank: {np.linalg.matrix_rank(rank1_matrix)}")

col_basis, left_null_basis = get_column_space_basis(rank1_matrix)

print(f"\nColumn space basis ({len(col_basis)} vector):")
for i, vec in enumerate(col_basis):
    print(f"  v{i+1} = {vec}")

print(f"\nLeft null space basis ({len(left_null_basis)} vectors):")
for i, vec in enumerate(left_null_basis):
    print(f"  u{i+1} = {vec}")

print()
verify_orthogonality(col_basis, left_null_basis)

Rank-1 Matrix:
[[1 2 3]
 [2 4 6]
 [3 6 9]]

Matrix rank: 1

Column space basis (1 vector):
  v1 = [1. 2. 3.]

Left null space basis (2 vectors):
  u1 = [-2.  1.  0.]
  u2 = [-3.  0.  1.]

Checking orthogonality (dot products):
  col_v1 · null_u1 = 0.0000000000 ✓
  col_v1 · null_u2 = 0.0000000000 ✓

All vectors are orthogonal!


True

## Example 2: Rank-1 Matrix

In [38]:
# Rank-2 matrix: third column is sum of first two
rank2_matrix = np.array([
    [1, 2, 3],
    [4, 5, 9],
    [7, 8, 15]
])

print("Rank-2 Matrix:")
print(rank2_matrix)
print(f"\nMatrix rank: {np.linalg.matrix_rank(rank2_matrix)}")

col_basis, left_null_basis = get_column_space_basis(rank2_matrix)

print(f"\nColumn space basis ({len(col_basis)} vectors):")
for i, vec in enumerate(col_basis):
    print(f"  v{i+1} = {vec}")

print(f"\nLeft null space basis ({len(left_null_basis)} vector):")
for i, vec in enumerate(left_null_basis):
    print(f"  u{i+1} = {vec}")

print()
verify_orthogonality(col_basis, left_null_basis)

Rank-2 Matrix:
[[ 1  2  3]
 [ 4  5  9]
 [ 7  8 15]]

Matrix rank: 2

Column space basis (2 vectors):
  v1 = [1. 4. 7.]
  v2 = [2. 5. 8.]

Left null space basis (1 vector):
  u1 = [ 1. -2.  1.]

Checking orthogonality (dot products):
  col_v1 · null_u1 = 0.0000000000 ✓
  col_v2 · null_u1 = 0.0000000000 ✓

All vectors are orthogonal!


True

## Example 1: Rank-2 Matrix